# EDA & Cleaning: MIMIC-IV Clinical Database Demo

Source: `mimic4_demo/icu_hosp/2.2/hosp/`. Newer real (demo-scale) ICU cohort, same role as the MIMIC-III demo: no genotype, but real prescriptions/labs/admissions.

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_columns", 100)
DATA_DIR = Path("..").resolve()
DECISIONS_PATH = Path("dataset_decisions.json")

def record_decision(name, verdict, reason, n_rows=None, n_cols=None, saved_to=None):
    decisions = {}
    if DECISIONS_PATH.exists():
        decisions = json.loads(DECISIONS_PATH.read_text())
    decisions[name] = {
        "verdict": verdict,
        "reason": reason,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "saved_to": saved_to,
    }
    DECISIONS_PATH.write_text(json.dumps(decisions, indent=2))
    print(f"Recorded decision for '{name}': {verdict}")


## File inventory: `mimic4_demo/` (two modules: `ed/2.2/ed/` with 7 tables, and `icu_hosp/2.2/` with `hosp/`: 21 tables and `icu/`: 9 tables)

**Used (all from `hosp/`):** `prescriptions.xlsx` (warfarin orders), `labevents.xlsx` plus `d_labitems.xlsx` (INR/PT results and item-ID lookup), `admissions.xlsx`, `diagnoses_icd.xlsx`, and `patients.xlsx` (age and gender, merged into the cleaned admissions table below).

**Not used:**
- The entire `ed/` module (`diagnosis`, `edstays`, `medrecon`, `pyxis`, `triage`, `vitalsign`): pre-admission ED encounter data, not inpatient warfarin management.
- The entire `icu/` module (`chartevents`, `datetimeevents`, `icustays`, `inputevents`, `outputevents`, `procedureevents`, `d_items`, `caregiver`, `ingredientevents`): ICU device/vitals charting, out of scope for a demo-scale warfarin/INR extraction.
- `hosp/omr.xlsx` (outpatient weight/height/BP, potentially useful, not incorporated), `hosp/pharmacy.xlsx`, `hosp/emar.xlsx`/`emar_detail.xlsx`, `hosp/poe.xlsx`/`poe_detail.xlsx` (more granular dispensing/order-entry records, redundant with `prescriptions.xlsx`), `hosp/d_hcpcs.xlsx`/`hcpcsevents.xlsx`/`drgcodes.xlsx` (billing codes), `hosp/d_icd_diagnoses.xlsx`/`d_icd_procedures.xlsx` (code dictionaries, not joined), `hosp/microbiologyevents.xlsx`, `hosp/procedures_icd.xlsx`, `hosp/provider.xlsx`, `hosp/services.xlsx`, `hosp/transfers.xlsx`.

In [ ]:
hosp = DATA_DIR / "usable/mimic4_demo/icu_hosp/2.2/hosp"
rx = pd.read_excel(hosp / "prescriptions.xlsx")
labevents = pd.read_excel(hosp / "labevents.xlsx")
d_labitems = pd.read_excel(hosp / "d_labitems.xlsx")
admissions = pd.read_excel(hosp / "admissions.xlsx")
diagnoses = pd.read_excel(hosp / "diagnoses_icd.xlsx")
patients = pd.read_excel(hosp / "patients.xlsx")
print("prescriptions:", rx.shape)
print("labevents:", labevents.shape)
print("admissions:", admissions.shape)
print("patients:", patients.shape)

In [ ]:
warfarin_rx = rx[rx["drug"].astype(str).str.contains("warfarin", case=False, na=False)]
n_warfarin_patients = warfarin_rx["subject_id"].nunique()
print(f"Warfarin prescription rows: {len(warfarin_rx)}, distinct patients: {n_warfarin_patients}")
warfarin_rx.head(3)

In [ ]:
inr_items = d_labitems[d_labitems["label"].astype(str).str.contains("INR", case=False, na=False)]
print(inr_items[["itemid", "label", "fluid", "category"]])

In [ ]:
inr_labs = labevents[labevents["itemid"].isin(inr_items["itemid"])]
print("INR lab rows found:", inr_labs.shape)
inr_labs.head(3)

## Cleaning
Keep only the warfarin-cohort subset of admissions, diagnoses, and INR labs, matching the MIMIC-III approach. Skip the large raw event tables not needed for modeling.

In [ ]:
warfarin_ids = warfarin_rx["subject_id"].unique()
if len(warfarin_ids) == 0:
    print("No warfarin patients found in this demo release, nothing to clean or save.")
else:
    adm_clean = admissions[admissions["subject_id"].isin(warfarin_ids)].copy()
    inr_clean = inr_labs[inr_labs["subject_id"].isin(warfarin_ids)]
    dx_clean = diagnoses[diagnoses["subject_id"].isin(warfarin_ids)]

    # Merge in age (at admission) and gender from patients.xlsx.
    adm_clean = adm_clean.merge(patients[["subject_id", "gender", "anchor_age", "anchor_year"]], on="subject_id", how="left")
    adm_clean["admittime"] = pd.to_datetime(adm_clean["admittime"])
    # MIMIC-IV convention: age at any admission = anchor_age + (admission_year - anchor_year), since real dates are shifted.
    adm_clean["age_years"] = adm_clean["anchor_age"] + (adm_clean["admittime"].dt.year - adm_clean["anchor_year"])
    adm_clean = adm_clean.drop(columns=["anchor_age", "anchor_year"])

    warfarin_rx.to_csv("cleaned/mimic4_warfarin_prescriptions_clean.csv", index=False)
    inr_clean.to_csv("cleaned/mimic4_inr_labs_clean.csv", index=False)
    adm_clean.to_csv("cleaned/mimic4_admissions_clean.csv", index=False)
    dx_clean.to_csv("cleaned/mimic4_diagnoses_clean.csv", index=False)
    print(f"Saved cleaned tables for {len(warfarin_ids)} warfarin patients")
    print("age_years summary:\n", adm_clean["age_years"].describe())

## Verdict

Depends on whether this demo release actually contains warfarin patients with INR labs (checked above). If yes: **KEEP**, same role as MIMIC-III (clinical-history validation, no genotype). If the cohort is empty or too small to be useful, mark **DROP** and rely on the MIMIC-III demo instead.

In [ ]:
verdict = "KEEP" if len(warfarin_ids) > 0 else "DROP"
reason = (
    f"Real (demo-scale) MIMIC-IV patients ({len(warfarin_ids)}) with warfarin Rx"
    + (" and INR labs; no genotype. Use for clinical-only validation alongside MIMIC-III."
       if len(warfarin_ids) > 0 else
       "; but too few/no warfarin+INR patients in this demo release to be useful.")
)
record_decision(
    "mimic4_demo", verdict=verdict, reason=reason,
    n_rows=int(len(warfarin_ids)), n_cols=None,
    saved_to="cleaned/mimic4_*_clean.csv" if len(warfarin_ids) > 0 else None,
)